### Library imports

In [ ]:
# Base libraries
import pandas as pd
import numpy as np
from ast import literal_eval
import os
import matplotlib.pyplot as plt
import pyprind
import copy
import scipy
from collections import defaultdict
import pickle

# Specialized libraries
import brightway2 as bw
import database_wide_monte_carlo as dbmc
import presamples as ps
from pelicun.base import Options, convert_to_MultiIndex
from pelicun.assessment import Assessment

# LIRIDE libraries
import pelicun_2_bw2 as p2b

### Function imports

In [ ]:
def get_C_matrices(demand, list_of_methods):
    """ Return a dict with {method tuple:cf_matrix} for a list of methods
    Uses a "sacrificial LCA" with exactly the same demand as will be used
    in the MonteCarloLCA
    """
    C_matrices = {}
    sacrificial_LCA = bw.LCA(demand)
    sacrificial_LCA.lci()
    for method in list_of_methods:
        sacrificial_LCA.switch_method(method)
        C_matrices[method] = sacrificial_LCA.characterization_matrix
    return C_matrices

def convert_act_to_score_S(act_UP, db_S, list_methods):
    """
    Create a database storing cumulative LCIA scores of specified LCIA methods for unit processes (UP).  
    """
    # Make sure activity and methods exist
    assert isinstance(list_methods, list), "Methods should be a list of methods"
    for m in list_methods:
        assert m in bw.methods, "{} is not a recognized method name"
    assert "Activity" in str(type(act_UP))
    # Create a new database for S datasets if one doesn't already exist
    if db_S not in bw.databases:
        bw.Database(db_S).register()
        
    # Make sure act_UP does not already exist in the S database : 
    try : 
        # Create a copy of the UP activity in the S database
        act_S = act_UP.copy(database=db_S, code=act_UP.key[1])
    except : 
        # If the dataset already exists : 
        act_S = bw.Database(db_S).get(act_UP.key[1])
    # Remove all exchanges except the production exchange
    for exc in act_S.technosphere():
        exc.delete()
    for exc in act_S.biosphere():
        exc.delete()
    # Create an LCA object to calculate LCIA scores
    lca = bw.LCA({act_UP:1})
    lca.lci()
    # Add LCIA scores for each method
    # The elementary flow key is ('biosphere3', method_abbreviation)
    for m in list_methods:
        new_exc = act_S.new_exchange()
        new_exc['input'] = ('biosphere3',
                            bw.Method(m).get_abbreviation()
                           )
        new_exc['output'] = act_S.key
        lca.switch_method(m)
        lca.lcia()
        new_exc['amount'] = lca.score
        new_exc['type'] = 'biosphere'
        new_exc.save()
    # Save the activity
    act_S.save()
        
    #cursor.close()
    return act_S

# Deaggragation functions
def standardize_technosphere(mc_ps) : 
    # Find total demand for aggregated datasets
    row_sums = mc_ps.technosphere_matrix.sum(axis=1)*(-1) # include a sign flip 
    # Row sums = 0 indicate the dataset is only an intermediate one (+1 on diagonal, -1 when consummed by another = 0)
    # Use the A/B LCA matrix format to isolate only the preaggregated datasets :
    reduced_sums = row_sums[np.nonzero(mc_ps.biosphere_matrix.sum(axis=0))[1]].T
    
    # Row sums = -1 indicate a non-triggered damage (+1 accounts for technosphere format of "1" as activity output) 
    filtered_sums = reduced_sums[:,np.where(reduced_sums!=-1)[1]]+1

    # Reduce the technosphere to match the row size of reduced_sum
    reduced_results = mc_ps.technosphere_matrix[np.nonzero(mc_ps.biosphere_matrix.sum(axis=0))[1],:]
    
    # Match the filtered sums row size
    filtered_results = reduced_results[np.where(reduced_sums!=-1)[1],:]


    # Standardize results relative to the total sum :
    sums_CSR_format = scipy.sparse.diags(1/filtered_sums.A.ravel())*(-1)
    standardized_technosphere = (sums_CSR_format @ filtered_results)

    # Remove production quantity columns
    standardized_technosphere.data[np.where(standardized_technosphere.data < 0)] = 0
    standardized_technosphere.eliminate_zeros()
    return standardized_technosphere

def deaggregate_results(standardized_technosphere, a_score) : 
    # Deaggregate results according to component ID
    deaggregated_results = (a_score @ standardized_technosphere)
    
    return deaggregated_results

def technosphere_col_IDs_to_act_name(LCA_obj,dependents):
    # Map Activities : 
    #--------------------------------------------------------------------
    # Initiate a storage variable
    all_unique_activities_and_their_keys = defaultdict(list)

    # Start iterating to link datasets with IDs : 
    for db in dependents :
        for activity in bw.Database(db) : 
            if activity.key in LCA_obj.activity_dict:
                all_unique_activities_and_their_keys[(f"{activity['reference product']}")].append(activity.key)

    # get the columns for all elementary processes : 
    all_unique_columns_and_their_activities = {
        LCA_obj.activity_dict[key[0]] : name 
        for name,key in all_unique_activities_and_their_keys.items() 
    }    
 
    return all_unique_columns_and_their_activities

### Data imports

In [ ]:
# Load the buildings to study
Buildings = pd.read_excel('Archetypes large building inv.xlsx')
display(Buildings.head(5))

In [ ]:
# Load the hazards
# Note : Wind speeds are defined in miles per hour ; relying on peak gust wind speed (PWS)
wind_df = pd.read_excel('Discretized_mp_v4.xlsx',header = [0,1],index_col = 0)
display(wind_df)

### Create a building inventory

In [ ]:
# Set the sample size 
sample_size = 10000

In [ ]:
# Uncertainty in the building portfolio : 
##########################################
# IPSOS survey : only 10% of Canadians have engaged in preparing their home to natural hazards
##########################################

individual_option_probabilities = {
    'SWP' : {
            '0':1,
            '1':0,
    },
    'R2W' : {
            'strap': 0.1,
            'tnail': 0.9,
    },
    'Nail_pattern' : {
            '6d':0.5,
            '6s':0,
            '8d':0.5,
            '8s':0,
    },
    'Rf_cover' : {
        'bur':0.5,
        'spm':0.5,
    },
    'Rf_condition' : {
        'por':0.05, # Assumption : only 5% of the houses would have deteriorated roof at the time of the hazard happening.
        'god':0.95,
    },
    'Wndow': {
        'low':0.55,
        'med':0.4,
        'hig':0.05,
    } 
}

In [ ]:
# Create a dataframe that will combine all the buildings according to their individual likelyhood to be a single value from the list

# Initialize variables
cmp_sample = pd.DataFrame()
archetypes_per_sample = np.array([])
loc_array = np.array([])
loc2guessed_archetype = {}
i = 0

# Map guessed archetypes to loc values (used for post-processing damages)
for id_val,name in enumerate(Buildings['Generic archetype']) : 
    loc2guessed_archetype[f'{id_val+1}'] = name

# Create the building inventory based on the archetypes : 
for row in Buildings.iterrows():    
    archetypes = literal_eval(row[1]['Potential archetypes'])
    list_archetypes = tuple((building,str(row[0]+1),'1') for building in archetypes) # Add story level and direction for pelicun internal interpretation

    guessed_archetype = row[1]['Guessed archetype']
    
    # Find position of the guesses : 
    archetype_inputs = guessed_archetype.split(".")
    guess_indexes = [(i,x) for i, x in enumerate(archetype_inputs) if x in individual_option_probabilities]

    ls_probs = []
    
    # Assign probabilities to the guesses, following the individual probability of each guess
    for archetype in list_archetypes :
        Prob = float(1)         
        archetype_features = archetype[0].split(".")        
        for guess in guess_indexes :
            prob_modifier = individual_option_probabilities[guess[1]][archetype_features[guess[0]]]            
            Prob = Prob*prob_modifier
        ls_probs.append(Prob)

    df_archetypes = pd.DataFrame(
        np.random.multinomial(1,ls_probs,size = sample_size), # Assign realisation archetypes, following probabilities
        columns = pd.MultiIndex.from_tuples(
        list_archetypes,names = ['cmp','loc','dir'])
    )


    df_archetypes = df_archetypes.loc[:, (df_archetypes != 0).any(axis=0)]
    archetypes_per_sample = np.append(archetypes_per_sample,df_archetypes.shape[1])
    
    cmp_sample= pd.concat([cmp_sample,df_archetypes.T],axis =0)

# Visualizations :
print('Displaying the number of archetype variations per sampled building :')
print(archetypes_per_sample)
display(cmp_sample.head(10))

# add units information - trivial here, but needed for pelicun anyway
cmp_sample['Units'] = ['ea',]*(int(archetypes_per_sample.sum()))
cmp_sample = cmp_sample.T

### Initialize the pelicun assessment

In [ ]:
# Initiate the assessment
PAL = Assessment({
    "PrintLog" : False,
    "Seed" : 415, 
    "NonDirectionalMultipliers" : {
        "ALL":1.0
    },
}) 

In [ ]:
# load the sample into the assessment
PAL.asset.load_cmp_sample(cmp_sample)

### Applying loads and damage sampling

In [ ]:
# Set some variables
MP_to_investigate = 'MP45'
MRIs = wind_df[MP_to_investigate]['Midpoint MRI'].to_list()

dmgs = {}
mapping_dict = {}
for v in range(1,int(archetypes_per_sample.sum())+1) : 
    mapping_dict[f'{v}'] = v

# load to the model
PAL.damage.load_damage_model([
    'PelicunDefault/fragility_DB_SimCenter_Hazus_HU.csv', #this is a table with the default Hazus data
])

# compute demands and damages to the structures for each MRIs :
for RI_ind in range(0,len(MRIs)) : 
    raw_demand = pd.DataFrame()
    raw_demand['Units'] = ['mph']
    
    raw_demand['Theta_0'] = wind_df[MP_to_investigate]['3 Second Gust Midpoint Wind speed (miles per hr)'].to_numpy()[RI_ind]
    raw_demand = pd.concat([raw_demand,]*len(cmp_sample.columns), ignore_index = True)
    raw_demand.index = [f'PWS-{str(loc+1)}-1' for loc in range(0,len(cmp_sample.columns))]
    #display(raw_demand)
    
    # load the demand distribution definition into pelicun
    PAL.demand.load_sample(raw_demand.T)
    PAL.demand.calibrate_model({"ALL":{"DistributionFamily":"empirical"}})    
    PAL.demand.generate_sample({"SampleSize":sample_size})
    
    # Compute damages
    PAL.damage.calculate()
    PAL.damage.save_sample()
    
    # Post-process the results : 
    dmg_samples = p2b.PBD_setup.clean_dmg_df(PAL)
    dmg_samples = dmg_samples.reorder_levels(['loc','cmp','dir','ds'],axis=1)
    dmg_samples = dmg_samples.rename(columns=mapping_dict, level=0)     
    dmg_samples = dmg_samples.sort_index(axis=1)

    # Store the results
    dmgs[f'{MRIs[RI_ind]}'] = dmg_samples    

In [ ]:
display_samples = False
if display_samples == True : 
    for key in dmgs.keys() : 
        print(f'Displaying 5 results out of {dmgs[key].shape[0]} for an MRI of {key} years.')
        display(dmgs[key].head(5))    

### Post processing : synthetize the amount of archetypes to individually damaged buildings

In [ ]:
# New dictionnary to store the post-processed results : 
dmgs_postprocessed = {}

for key in dmgs.keys() :
    df_key = dmgs[key].copy()    
    
    # Drop the multi-archetype naming convention/loc - they do not influence the loss modeling (LCA)
    df_key = df_key.groupby(level = [0,2,3],axis =1).sum()

    # Assign a corresponding guessed archetype to each loc :
    df_names = list(df_key.columns.names) # get current names of index levels
    df_names.insert(0,'cmp') # re-introduce cmp as a index level value. Note the cmp-loc-dir order is preserved (for compatibility with component-based LCA computations)
    df_key.columns = pd.MultiIndex.from_tuples([(loc2guessed_archetype[str(k[0])],k[0],k[1],k[2]) for k in df_key.columns],names =df_names )
       
    # Store the results
    dmgs_postprocessed[f'{key}'] = df_key     

In [ ]:
display_samples = False
output_samples = False
if display_samples == True : 
    for key in dmgs_postprocessed.keys() : 
        print(f'Displaying 5 results out of {dmgs_postprocessed[key].shape[0]} for an MRI of {key} years.')
        display(dmgs_postprocessed[key].head(5)) 
        
if output_samples == True :         
    with pd.ExcelWriter(f'{MP_to_investigate}-dmg_samples.xlsx') as writer : 
        for key in dmgs_postprocessed.keys():
            pd.DataFrame(dmgs_postprocessed[key]).to_excel(writer,sheet_name=str(key))    

### Indicate a mapping of potential losses relating to these buildings

In [ ]:
#  Find a mapping between archetypes and their potential claddings : 
Buildings['Cladded base types'] = Buildings['base_type']+'-'+ Buildings['cladding']
unique_types = Buildings.drop_duplicates('Cladded base types')

groups = unique_types.groupby('cladding')
unique_archetype_claddings = {}
archetypes_map = {}

for group in groups : 
    archetypes_ = group[1]['Potential archetypes'].apply(literal_eval).map(lambda x: x[0])
    generic_archetypes = group[1].loc[archetypes_.index,'Generic archetype']
    
    for k in archetypes_ : 
        if k not in unique_archetype_claddings.keys() : 
            unique_archetype_claddings[k] = [group[0]]
        else : 
            unique_archetype_claddings[k].append(group[0])
    
    # Save a mapping between generic archetypes and specific ones : 
    archetypes_df = pd.concat([archetypes_,generic_archetypes],axis = 1) 
    for row in archetypes_df.iterrows() : 
        if row[1]['Potential archetypes'] not in archetypes_map.keys() : 
            archetypes_map[row[1]['Potential archetypes']] = row[1]['Generic archetype']
    
unique_archetype_claddings

In [ ]:
# we also need to re-define the loss map to have all archetypes included
unique_archetypes = unique_archetype_claddings.keys()

# all of the drivers are damage quantities (rather than EDP or IM intensities)
# so we need to prepend 'DMG-' to the component names to tell pelicun to look for the damage of these components
drivers = [f'DMG-{cmp}' for cmp in unique_archetypes]

# loss models are archetype-specific, identified by the archetype name
loss_models = unique_archetypes 

# Assemble a DataFrame with the mapping information 
# The column name identifies the type of the consequence model
loss_map = pd.DataFrame(loss_models,columns = ['BldgRepair'],index = drivers)

loss_map

In [ ]:
PAL.bldg_repair.load_model(['PelicunDefault/bldg_repair_DB_SimCenter_Hazus_HU.csv'],
    loss_map.iloc[:,:1])

# check the parameters assigned to the archetype
loss_parameters = PAL.bldg_repair.loss_params
display(loss_parameters)

### Identifying the environmental impacts

##### Adapting damage outputs to account for LCI requirements

In [ ]:
# Features from buildings to include from an LCA perspective
features = ['cladding',]

extended_dmgs = copy.deepcopy(dmgs_postprocessed)

for key in extended_dmgs.keys():
    dmg_df = extended_dmgs[key]
    all_house_types = extended_dmgs[key].columns.get_level_values(0).tolist()
    all_dir = extended_dmgs[key].columns.get_level_values(1).tolist()
    
    featured_houses = []
    
    for h,d in list(zip(all_house_types,all_dir)) :
        House_type = h
        for feature in features :
            House_type = House_type +'-'+ Buildings.loc[d-1,feature]
        featured_houses.append(House_type)
        
   # Use the featured houses as the new cmp index level 
    old_idx = dmg_df.columns.to_frame()    
    old_idx.insert(0, 'cmp_new', featured_houses)
    dmg_df.columns = pd.MultiIndex.from_frame(old_idx)
    dmg_df.columns.rename(['cmp','old_cmp','loc','dir','ds'], inplace= True)
    dmg_df.columns = dmg_df.columns.droplevel(1)
    
    extended_dmgs[key] = dmg_df

In [ ]:
# list unique entries per feature :
unique_features = {}
for feature in features : 
    unique_features[feature] = Buildings[feature].unique().tolist()    

##### Create a more comprehensive list of LCIs 

In [ ]:
# Currently available damages are tied to generic archetypes, not all the LCA ones. Damage states should therefore
# be re-assigned to the generic archetype names, inclusive of claddings.
base_LCIs_to_list = list(p2b.PBD_setup.mapping_DS_to_LCI(loss_parameters))

LCI_drivers = {}

# Obtain all dmg states the archetypes are likely to trigger :
for k in base_LCIs_to_list :
    if k[:-4] not in LCI_drivers.keys() : 
        LCI_drivers[k[:-4]] = [k[-4:]]
    else : 
        LCI_drivers[k[:-4]].append(k[-4:])


# Obtain a list with all generic archetypes with their relevant features (cladding) and damages
LCIs_to_list = []

for driver in LCI_drivers :
    main_ds = archetypes_map[driver]
    for cladding in unique_archetype_claddings[driver]:
        for dmgs in LCI_drivers[driver] :
            LCIs_to_list.append(main_ds+'-'+cladding+dmgs)

### Create an extended loss map 

In [ ]:
# Not all archetypes LCIs are modeled (i.e. gable vs hip is currently not implemented)
# Nevertheless, a simplified LCI is still reasonable : the extended loss map should point dmgs to the closest matching LCI

In [ ]:
# All enabled archetypes in the damages : 
all_archetypes = []
for key in extended_dmgs.keys():
    assessment = extended_dmgs[key]
    all_archetypes.extend(assessment.columns.get_level_values(0).to_list())
    
all_archetypes = set(all_archetypes)

# All archetypes included within the LCI : 
archetypes_with_LCI = set([x[:-4] for x in LCIs_to_list])

# LCI gap :
differences = list(set(all_archetypes) - set(archetypes_with_LCI))

# For this assessment, these distinctions will be ignored in the loss map :
LCI_remap = {
    'hip':'gab', # LCI currently does not have any distinction for roof shape.
    'std':'no' # LCI currently does not have any distinction for a garage or not.
}

remapped_differences = {}
for a in differences : 
    b = copy.copy(a)
    for k in LCI_remap.keys() : 
        if k in a : 
            b = b.replace(k,LCI_remap[k])

    remapped_differences[a] = b
            
            
# Build the extended loss map :           
extended_loss_map = {}
extended_indexes = []
extended_cols = []

# Start by filling the map with the archetypes with an existing LCI match :
for a in archetypes_with_LCI : 
    extended_indexes.append('DMG-'+a)
    extended_cols.append(a)
    
# And then add the mapping to LCIs :
for k,v in remapped_differences.items() : 
    extended_indexes.append('DMG-'+k)
    extended_cols.append(v)

# Convert both lists to a dataframe
extended_loss_map = pd.DataFrame(extended_cols, columns = ['BldgRepair'],index = extended_indexes)
display(extended_loss_map)

#### Preparing the LCA

In [ ]:
#================================================
#        Migrate damage states to Brightway2
#================================================
# Variables
project_name = 'OS2 - ei312 - Portfolio of buildings'     # Will set up the project under this name.


eco_XX_db = 'ecoinvent_3.12_cutoff'             # Name of the desired ecoinvent database.
fpeiXX = r'D:\Doctorat\10 - ecoinvent\ecoinvent 3_12 cut_off\datasets' # local path to ecoinvent datasets
background_loss_db_name = 'Building repairs'      # Unique name for the database storing the damage states.
foreground_loss_db_name = 'DS_LCI-Portfolio'
#================================================
# Setting up the project and background databases
p2b.LCA_setup.bw2_launch(project_name)
ei_CO = p2b.LCA_setup.get_ei_db(eco_XX_db,filepath = fpeiXX) # Getting the technosphere matrix (ecoinvent Cut-off database)
#================================================
# Mapping damage outputs to LCA inputs : 
# Recall : loss_parameters identifies damageable assets.
p2b.LCA_setup.add_damage_states_to_LCI(background_loss_db_name,LCIs_to_list)

#### Quickly copy all the basic technosphere/elementary flows from a reference house to all others in the database (shortcut to creating numerous house datasets) 
* Note : Before loading the damage states to presamples, scalling ratios will need to be applied (not all houses are the same amount of square meters!)

In [ ]:
# Re-select the project of interest
bw.projects.set_current(project_name)

# Archetype from which to copy flows from :
base_datasets = {
    'W.SF':{
        'area' :3010, # gross floor area, in squared feets, over the two stories
        'storeys':2,
        'coverage':['W.SF','W.MU',],
        'storage':'Reference_buildings',  # Name of database containing the reference dataset
        'code DS1':'bf4805eb0e78417196e02b92f2559506', # local database code (convenient for modeling with the activity-browser)
        'code DS2':'a0485e05a35c42e1b30b0039221e38bd',
        'code DS3':'71b1d04068394bdfa1d3d0b4a2d37586',
        'code DS4':'9fe824b943d54bd4bc1608b90e6ec5fa',
    },
    'C.ERB':{
        'area':28800,
        'storeys':4,
        'coverage':['C.ER',],
        'storage':'Reference_buildings',
        'code DS1':'cd9a9365683a490e832f89dde8e00a19', # local database code (convenient for modeling with the activity-browser)
        'code DS2':'2eb0551caddc447194ab0620b2a6e1c4',
        'code DS3':'6cbde5580c9d4926aa0d564b7ff7a92f',
        'code DS4':'4831c5d14ab4493289b60aad28ab94b3',        
    },
    'M.SF':{
        'area':3010,
        'storeys':2,
        'coverage':['M.SF',],
        'storage':'Reference_buildings',        
        'code DS1':'32086a029e714292bc8659c3e06683a8', # local database code (convenient for modeling with the activity-browser)
        'code DS2':'1c3f306fba354ccda5a6d5f22567e437',
        'code DS3':'f9a69795b324436a9996410f04ef74a8',
        'code DS4':'1fe86eebb82f4fe3a97aebd9bb4fd0fa',
        
    }
}

# Reverse the mapping of base datasets for its coverage : 
coverage_map = {}
for key, val in base_datasets.items() : 
    for i in val['coverage']: 
        coverage_map[i] = key
        
print(coverage_map)

In [ ]:
# Build a mapping of the scaling ratios :
# lets store to a dictionary scaling ratios
loc2scaling_data = {}

# Generate the scaling ratio data : 
for row in Buildings.iterrows() : 
    plan_area = row[1]['PlanArea']
    stories = row[1]['NumberOfStories']
    
    matching_LCA_archetype = coverage_map[row[1]['base_type'][0:4]]
    ref_archetype_area = base_datasets[matching_LCA_archetype]['area']
    ref_archetype_storeys = base_datasets[matching_LCA_archetype]['storeys']
    

    # Store tuples in a dictionary that will help provide adequate scaling factors (DS1-DS3 are plan area dependent)
    # DS4 is also plan area dependent, but the number of stories increases the true total plan area.    
    loc2scaling_data[f'{row[0]+1}'] = (plan_area/(ref_archetype_area/ref_archetype_storeys),(plan_area*stories)/(ref_archetype_area))

In [ ]:
ref_LCIs = {}

# Iterate through the LCIs in the background database : 
for ds in bw.Database(background_loss_db_name) : 
    if len(ds.exchanges())  <= 1  : #If a dataset is empty, fill with data stemming from the reference buildings
 
        # Identify the reference dataset to copy :
        ref_base_dataset = coverage_map[ds['name'][0:4]]
        dmg_state = ds['name'][-3:]
        ref_building_name = ref_base_dataset + ' ' + dmg_state
        ref_db = base_datasets[ref_base_dataset]['storage']
        
        # Access the reference dataset :
        ref_building_code = (base_datasets[ref_base_dataset]['storage'],base_datasets[ref_base_dataset]['code '+ dmg_state])
        ref_building_ds = bw.get_activity(ref_building_code)
        
        # Sort out features unrelevant to this building :
        enabled_features = ds['name'].split(' ')[0].split('-')[1:]
        disabled_features = []        
        for pair in list(zip(features,enabled_features)) :
            available_features = copy.deepcopy(unique_features[pair[0]])
            if pair[1] in available_features: available_features.remove(pair[1])
                # Ensure the feature names are compatible with the LCI naming
            available_features = [x+ ' '+ pair[0] for x in available_features]
            disabled_features.extend(available_features)
        disabled_features = [x.lower() for x in disabled_features]

    
        exchanges_to_track = []        
        # Run through the exchanges. 
        for exc in ref_building_ds.exchanges() :
            # Avoid features which do not match the building :            
            if any(x in exc.input['name'] for x in disabled_features) :
                continue
            
            # If the exchange is not a production flow, store the exchanges as a list of tuples :
            if (exc.input != ref_building_ds) :
                # Manually identify the flow type (other fields all have their own methods already) :
                if exc.input['type']=='process' : 
                    exc_type = 'technosphere'
                if exc.input['type']=='emission' : 
                    exc_type = 'biosphere'
                # Make a list of tuples for each flow   
                exchanges_to_track.append((exc.input.key,exc.amount,exc.unit,exc_type))
        # Fill the dictionary with the list of tuples :
        ref_LCIs[ds['name']] = exchanges_to_track 

# Lets start iterating over the database to fill empty datasets:
for ds in bw.Database(background_loss_db_name) :
    # Find if a dataset is empty. If so, it should be populated automatically
    if len(ds.exchanges()) <= 1 :
        # Mark the relative progress :
        # Identify the reference dataset to copy :
        ref_base_dataset = coverage_map[ds['name'][0:4]]
        dmg_state = ds['name'][-3:]
        ref_building_name = ref_base_dataset + ' ' + dmg_state
    
    
        print(f'Adding exchanges to {ds} from {ref_building_name}')        
        
        # Add the new exchanges to the datasets : 
        for exc in ref_LCIs[ds['name']] : 
            ds.new_exchange(input = exc[0],amount=exc[1],unit=exc[2],type=exc[3]).save()
            ds.save() 

#### DBMC, balancing &  preaggregating datasets

In [ ]:
target_iterations = 500 # batch process, can be called several times until desired total count is achieved.
target_directory = r'D:\Doctorat\08 - Brightway2 - OpenSees - Pelicun\Prototype\dbmc_ei312'
cpus_available = os.cpu_count()
print(f'There are {cpus_available} cpus available on this machine.')
print(f'Filepath to stored simulations : {target_directory}')

##########################
# Saving to disk variables : 
include_inventory = True
include_supply = False
include_matrices = False
include_extended_inventory = False
balance_water = True
balance_land_use = True

delete_temps = True

In [ ]:
# Tel DBMC what project and databases to work with : 
bw.projects.set_current(project_name)
target_db = 'Building repairs'
new_db_name = 'Building presampled repairs'
target_db_size = len(bw.Database(target_db))
print(f'The target database holds {target_db_size} datasets.')

In [ ]:
%%time
# Task DBMC to prepare the inventory samples : 
dbmc.sample_generation.generate_samples_job(project_name,target_db,target_iterations,cpus_available-2,
                                            target_directory,
                                            include_inventory=include_inventory,
                                            include_extended_inventory = include_extended_inventory,
                                            include_supply = include_supply,
                                            balance_water = balance_water,
                                            balance_land_use = balance_land_use
                                           )

In [ ]:
%%time
# Task DBMC to clean up corrupted calculations
dbmc.clean_jobs(target_directory,target_db,target_db_size,include_supply=include_supply)

In [ ]:
%%time
# Recover individual job samples (usefull when the MC sampling has been split)
dbmc.concatenate_within_jobs(target_directory,target_db,include_inventory,include_supply,include_matrices,cpus_available-2,delete_raw_files = True)

In [ ]:
%%time
# Bug fixes : If more than one job to concatenate, it currently fails (translator function is off and needs fixing).
# If concatenation crashes on Excel files generation, might be due to malformed datasets (missing fields such as "production amount")
dbmc.concatenate_across_jobs(target_directory,target_db,project_name,include_inventory,include_supply,include_matrices,delete_temps)

#### Select the desired LCIA methods to preaggregate for

In [ ]:
%%time
# Applying LCIA methods : 

# 1st : Highlight which LCIA methods should be run : 
# Currently, a simple trick is to get the "methods description file" and copy the column headers + rows of desired LCIA methods.
dbmc.create_list_methods_from_xlsx(target_directory,target_db,'selected_LCIA_methods.xlsx')

In [ ]:
%%time
dbmc.dispatch_LCIA_calc_to_workers(target_directory,project_name,target_db,cpus_available-2,'selected_LCIA_methods')

In [ ]:
# Enable aggregated datasets
# List the methods requiring to be modified
# Make a quick script to retrieve the "selected_LCIA_methods" excel file? Currently, just copy the output from dispatch_LCIA_calc_to_workers
list_of_methods = [('IPCC 2021', 'climate change', 'global warming potential (GWP100)'), ('ReCiPe 2016 v1.03, endpoint (H)', 'total: ecosystem quality', 'ecosystem quality'), ('ReCiPe 2016 v1.03, endpoint (H)', 'total: human health', 'human health'), ('ReCiPe 2016 v1.03, endpoint (H)', 'total: natural resources', 'natural resources'),('IMPACT World+ Damage 2.0.1', 'Ecosystem quality', 'Total ecosystem quality'),('IMPACT World+ Damage 2.0.1', 'Human health', 'Total human health')]

In [ ]:
# Add 1 elementary flow to the biosphere, per LCIA method 

from bw2agg.scores import add_unit_score_exchange_and_cf 

for a_method in list_of_methods :
    # Lets only do it once :
    if not [ef for ef in bw.Method(a_method).load() if bw.Method(a_method).get_abbreviation() in ef[0]] :
        add_unit_score_exchange_and_cf(a_method)
    else :
        print([ef for ef in bw.Method(a_method).load() if bw.Method(a_method).get_abbreviation() in ef[0]])   

In [ ]:
# This step may sometimes crash (cursor related error)
for ds in bw.Database(target_db):
    ds_S = convert_act_to_score_S(ds,new_db_name,list_of_methods)
    
# Ensure the new database ONLY contains datasets matching the "Building_repairs" database :     
ls1 = [ds['code'] for ds in bw.Database(target_db)]
ls2 = [ds['code'] for ds in bw.Database(new_db_name)]
diffs = [x for x in ls2 if x not in ls1]

if len(diffs) != 0 :
    print(f'The following datasets are getting removed from the new database : {diffs}')
    for ds in diffs : 
        bw.Database(new_db_name).get(ds).delete()        
    
# Quick Sanity check :
for ds in bw.Database(new_db_name) : 
    print(ds)
    print(f'Technosphere : {[*ds.technosphere()]}')
    print(f'Biosphere : {[*ds.biosphere()]}')

#### Presampling layer 1 - The DBMC layer

In [ ]:
# Path leading to DBMC final results :
lcia_fp = os.path.join(target_directory,target_db,"results\LCIA")

# Variable to load DBMC results
arrays_list = []
# Simulatenously list the datasets to which individual simulations belong : 
indices = []

# Start extracting DBMC results, matching preaggregrated LCIA results (as fake elementary flows) with LCIs 
for m in list_of_methods : 
    for act in bw.Database(new_db_name) : 
        arrays_list.append(np.load(os.path.join(lcia_fp, bw.Method(m).get_abbreviation(), act.key[1]+'.npy')).reshape(-1,1))
        indices.append(
            (
            ('biosphere3', bw.Method(m).get_abbreviation()),
                act.key,
                'biosphere'
            ))

# Stack the arrays
samples = np.hstack(arrays_list)

# Transpose
samples = samples.T
print(f'The shape of the loaded datapackage is : {samples.shape}')

# Final formatting :  
label = 'biosphere'
agg_matrix_data = [(samples, indices, label)]

# Loading to presamples : 
ps_id, ps_fp = ps.create_presamples_package(matrix_data=agg_matrix_data)
ps_fp


#### Presample layer 2 - The damage layer (scalled if necessary)

In [ ]:
# Create individual foreground databases assessing the MRI scenarios :
pp_paths = []

# Store the scaled damages :
stored_scaled_damages = {}

check_scaled_dmgs = False

for run in extended_dmgs.keys() : 
    print('RUN ID : ',run)
    foreground_name = f'{foreground_loss_db_name}-MRI-{run}'

    #scaling function to consider the number of square meters which wasn't implemented for individual houses
    scaled_dmgs = extended_dmgs[run].copy()
    for loc2scale in loc2scaling_data.keys() : 
        ratio1 = loc2scaling_data[loc2scale][0]
        ratio2 = loc2scaling_data[loc2scale][1]     

        # Find out WHICH columns should be scaled to what amount : 
        scale_list_1 = []
        scale_list_2 = []  
        scale_list_3 = []  
        scale_list_4 = []          
        # Try and scale a single 'loc'. If this fails, it means there is no dmg triggered at all. Thus, skip
        try :
            lst = [col for col in scaled_dmgs.loc[:,(slice(None),[int(loc2scale)],slice(None),slice(None))]]  
            for val in lst : 
                if val[-1] == '4' :  
                    scale_list_4.append(val[-1])
                elif val[-1]=='3':
                    scale_list_3.append(val[-1]) 
                elif val[-1]=='2':
                    scale_list_2.append(val[-1])
                else :
                    scale_list_1.append(val[-1]) 


            if len(scale_list_1) != 0 : 
                scaled_dmgs.loc[:,(slice(None),[int(loc2scale)],slice(None),scale_list_1)] = scaled_dmgs.loc[:,(slice(None),[int(loc2scale)],slice(None),scale_list_1)].mul(ratio1)
            if len(scale_list_2) != 0 : 
                scaled_dmgs.loc[:,(slice(None),[int(loc2scale)],slice(None),scale_list_2)] = scaled_dmgs.loc[:,(slice(None),[int(loc2scale)],slice(None),scale_list_2)].mul(ratio1)                
            if len(scale_list_3) != 0 : 
                scaled_dmgs.loc[:,(slice(None),[int(loc2scale)],slice(None),scale_list_3)] = scaled_dmgs.loc[:,(slice(None),[int(loc2scale)],slice(None),scale_list_3)].mul(ratio1)
            if len(scale_list_4) != 0 : 
                scaled_dmgs.loc[:,(slice(None),[int(loc2scale)],slice(None),scale_list_4)] = scaled_dmgs.loc[:,(slice(None),[int(loc2scale)],slice(None),scale_list_4)].mul(ratio2)             


        except : 
            continue

    stored_scaled_damages[run] = pd.DataFrame(scaled_dmgs) 
    
    
    ## workaround to avoid sqlite3 vacuum error :
    #if foreground_name in bw.databases: 
    #    del bw.databases[foreground_name]
    
    # Create linking between pre-aggregated damage states and foreground inventories : 
    p2b.LCA_setup.linking_DS_to_LCI(foreground_name,new_db_name,scaled_dmgs,extended_loss_map)
    # Whilst there, also pre-set presamples :
    pp_paths.append(p2b.LCA_setup.presamples_setup(foreground_name,new_db_name,scaled_dmgs,extended_loss_map)) 
    
    
if check_scaled_dmgs == True : 
    with pd.ExcelWriter(f'{MP_to_investigate}-scaled_dmg_samples.xlsx') as writer : 
        for key in stored_scaled_damages.keys():
            stored_scaled_damages[key].to_excel(writer,sheet_name=key) 

#### Run the LCA calculations

##### Detailed results

In [ ]:
%%time

deaggregate_outputs = True

if deaggregate_outputs == True :
    # Run the LCA
    # Calculation setups :
    iterations = 10000 # How many Monte Carlos to sample?
    LCIA_methods = list_of_methods

    # Storage variable :
    deaggregated_losses = {}

    # Start iterating the MRIs losses : 
    for j,run in enumerate(extended_dmgs.keys()) :

        # Preparing the reference flow and matrices :
        foreground_name = f'{foreground_loss_db_name}-MRI-{int(run)}' 
        demand = {(foreground_name, 'Structure losses'): 1} # To populate the demand vector
        if j == 0 : # Ensure this is only ran on the first iteration
            C_matrices = get_C_matrices(demand,LCIA_methods) # Pre-load the characterization matrices

        # Pre-define a matrix to store the MC results : 
        technosphere_data_count = len(bw.Database(new_db_name)) + len(bw.Database(foreground_name))

        mc_scores = np.empty(shape=[iterations,len(LCIA_methods)*technosphere_data_count]) # Pre-organize a matrix to store results.

        # Instantiate an object for MonteCarlos : 
        mc_ps = bw.MonteCarloLCA(
            demand,
            presamples=[

                        ps_fp,
                        pp_paths[j],
                       ]
        )

        # Launch calculations :
        print(f'Assessing an MRI of {MRIs[j]}, with {iterations} simulations.')

        for iteration in pyprind.prog_bar(range(iterations)) : 
            lci = next(mc_ps)
            stndzd_foreground = standardize_technosphere(mc_ps)
   
            for i,m in enumerate(LCIA_methods):
                score = (C_matrices[m]*mc_ps.inventory)

                # Sometimes, deaggregating trips numerical roundoff issues, which interferes with matrix shapes
                try :
                    deaggregated_score = deaggregate_results(stndzd_foreground,score.data)
                    if (score.sum()-deaggregated_score.sum())/score.sum()*100 > 0.5 : 
                        print('Warning. Some deaggregated results seem to have over 0.5% difference relative to original ones')            
                except :
                    cleansed_score = score.data[~np.isclose(score.data,0)]
                    deaggregated_score = deaggregate_results(stndzd_foreground,cleansed_score)
                    if (score.sum()-deaggregated_score.sum())/score.sum()*100 > 0.5 : 
                        print('Warning. Some deaggregated results seem to have over 0.5% difference relative to original ones')

                # Save the deaggregated MC result
                mc_scores[iteration,i*technosphere_data_count:(i+1)*technosphere_data_count] = deaggregated_score
        
        # Find the corresponding component names
        dependents = bw.Database(foreground_name).find_dependents()
        dependents.append(foreground_name)

        col_ID_map = technosphere_col_IDs_to_act_name(mc_ps,dependents)        

        # Build a pandas DataFrame with the results and relevant column names:
        col_tuples = []
        for i,m in enumerate(LCIA_methods) :
            for j in range(technosphere_data_count) : # enables to track the matrix column ID
                # Component ID
                cmp_id = col_ID_map[j].partition('cmp')[-1].split('-')[0]
                if cmp_id == '': 
                    cmp_id = 0
                # Damage state number
                DS = col_ID_map[j].partition('DS')[-1]        
                if DS == '': 
                    DS = 0

                col_tuples.append((m,int(cmp_id),int(DS),j+1,col_ID_map[j]))

        df = pd.DataFrame(mc_scores, columns = pd.MultiIndex.from_tuples(col_tuples))
        # Format for ease of reading :
        df = df.sort_index(axis = 1,level= 0)
        df = df.loc[:, (df != 0).any(axis=0)]
        df.columns = df.columns.droplevel(3)        
        
        
        # Force reset the sequential indexer
        mc_ps.presamples.reset_sequential_indices()
        # Clear the variable before giving it another run
        del mc_ps 
        deaggregated_losses[run] = df

In [ ]:
deaggregate_outputs_to_excel = True
if deaggregate_outputs_to_excel == True :
    a_key = list(deaggregated_losses.keys())[0]
    size = deaggregated_losses[a_key].shape[0]
    
    if size > 5000 : 
        print('Saving deaggregated results as a pickle file.')
        with open(f'{MP_to_investigate}-deagg_samples.pickle','wb') as handle:
            pickle.dump(deaggregated_losses, handle, protocol=pickle.HIGHEST_PROTOCOL)        
    else : 
        print('Saving as deaggregated results an Excel file.')
        with pd.ExcelWriter(f'{MP_to_investigate}-deagg_samples.xlsx') as writer : 
            for key in deaggregated_losses.keys():
                df = deaggregated_losses[key]
                df.mask(df.eq(0)).to_excel(writer,sheet_name=str(key))

###### Aggregated results

In [ ]:
%%time
# Run the LCA

# Calculation setups :
#iterations = 5000 # How many Monte Carlos to sample?
iterations = sample_size
LCIA_methods = list_of_methods

# Storage variable :
losses = {}

# Start iterating the MRIs losses : 
for j,run in enumerate(dmgs_postprocessed.keys()) :
    
    # Preparing the reference flow and matrices :
    foreground_name = f'{foreground_loss_db_name}-MRI-{int(run)}' 
    demand = {(foreground_name, 'Structure losses'): 1} # To populate the demand vector
    if j == 0 : # Ensure this is only ran on the first iteration
        C_matrices = get_C_matrices(demand,LCIA_methods) # Pre-load the characterization matrices
    mc_scores = np.empty(shape=[len(LCIA_methods), iterations]) # Pre-organize a matrix to store results.

    # Instantiate an object for MonteCarlos : 
    mc_ps = bw.MonteCarloLCA(
        demand,
        presamples=[

                    ps_fp,
                    pp_paths[j],
                   ]
    )

    # Launch calculations :
    print(f'Assessing an MRI of {MRIs[j]}, with {iterations} simulations.')

    for iteration in pyprind.prog_bar(range(iterations)) : 
        lci = next(mc_ps)   
        for i,m in enumerate(LCIA_methods):
            mc_scores[i, iteration] = (C_matrices[m]*mc_ps.inventory).sum()

    # Force reset the sequential indexer
    mc_ps.presamples.reset_sequential_indices()
    # Clear the variable before giving it another run
    del mc_ps 
    losses[run] = mc_scores


##### Visusal outputs for aggregated results

In [ ]:
# Visual outputs
fig, axs = plt.subplots(len(MRIs), len(LCIA_methods), figsize=(20, 40))

for MRI,results in enumerate(losses.keys()) :
    for val in range(len(LCIA_methods)) :
        
        arr = losses[results].T[:,val]
        lft_bnd = np.percentile(losses[results].T[:,val],0.3)
        rght_bnd = np.percentile(losses[results].T[:,val],99.7)
        
        arr = arr[arr>=0]
        arr = arr[arr<rght_bnd]
     
        # Format the results
        _, bin_edges = np.histogram(arr,density = True)
        
        
        # Plot the results
        axs[MRI,val].hist(arr,bins = bin_edges,alpha = 0.3)  
            
        # Add figure descriptions :
        if MRI == 0 : 
            axs[MRI,val].set_title(f'{LCIA_methods[val][0]}-{LCIA_methods[val][2]}')
        if val == 0 :
            axs[MRI,val].set_ylabel(f'Count at MRI {MRIs[MRI]}')
            
        if MRI == len(losses.keys())-1 : 
            axs[MRI,val].set_xlabel(bw.Method(LCIA_methods[val]).metadata['unit'])
        

check_outputs = True
if check_outputs == True : 
    with pd.ExcelWriter(f'{MP_to_investigate}-MC_results.xlsx') as writer : 
        for MRI, results in enumerate(losses.keys()):
            pd.DataFrame(losses[results].T,columns = LCIA_methods).to_excel(writer,sheet_name=str(MRIs[MRI]),index=False)

#### Selected Visual outputs for aggregated results

In [ ]:
selected_LCIA_2display = [0,4,5,]
selected_MRIs = [19,753,407160]
units = ['kg CO$_{2-eq}$','PDF/(m$^2$-year)','DALY' ]

# Visual outputs
fig, axs = plt.subplots(len(selected_MRIs), len(selected_LCIA_2display), figsize=(16, 15))

for i, MRI in enumerate(selected_MRIs) : 
    for j, LCIA_id in enumerate(selected_LCIA_2display):

        arr = losses[str(MRI)].T[:,LCIA_id]
        
        lft_bnd = np.percentile(arr,0.1)
        rght_bnd = np.percentile(arr,99.9)
        
        arr = arr[arr>=0]
        arr = arr[arr<rght_bnd]    
    
        # Format the results
        _, bin_edges = np.histogram(arr,density = True)    


        # Plot the results
        axs[i,j].hist(arr,bins = bin_edges,alpha = 0.3) 
 
        # Add figure descriptions :
        if i == 0 : 
            axs[i,j].set_title(f'{LCIA_methods[LCIA_id][0]} - {LCIA_methods[LCIA_id][1]}')
        if j == 0 :
            axs[i,j].set_ylabel(f'Count at MRI {MRI}')
            
        if i == len(selected_MRIs)-1 : 
            axs[i,j].set_xlabel(units[j])
